# Topics in Quantitative Finance - Homework 3 Solution

Assigned: Friday, July 31, 2026.
Due: **Monday, August 3, 2026** by 1PM. 

Late homework **will not be accepted**.

$$
\newcommand{\supp}{\mathrm{supp}}
\newcommand{\E}{\mathbb{E} }
\newcommand{\Eof}[1]{\mathbb{E}\left[ #1 \right]}
\def\Cov{{ \mbox{Cov} }}
\def\Var{{ \mbox{Var} }}
\newcommand{\1}{\mathbf{1} }
\newcommand{\PP}{\mathbb{P} }
\newcommand{\Pof}[1]{\mathbb{P}\left[ #1 \right]}
%\newcommand{\Pr}{\mathrm{Pr} }
\newcommand{\QQ}{\mathbb{Q} }
\newcommand{\RR}{\mathbb{R} }
\newcommand{\DD}{\mathbb{D} }
\newcommand{\HH}{\mathbb{H} }
\newcommand{\spn}{\mathrm{span} }
\newcommand{\cov}{\mathrm{cov} }
\newcommand{\sgn}{\mathrm{sgn} }
\newcommand{\HS}{\mathcal{L}_{\mathrm{HS}} }
%\newcommand{\HS}{\mathrm{HS} }
\newcommand{\trace}{\mathrm{trace} }
\newcommand{\LL}{\mathcal{L} }
%\newcommand{\LL}{\mathrm{L} }
\newcommand{\s}{\mathcal{S} }
\newcommand{\ee}{\mathcal{E} }
\newcommand{\ff}{\mathcal{F} }
\newcommand{\hh}{\mathcal{H} }
\newcommand{\bb}{\mathcal{B} }
\newcommand{\dd}{\mathcal{D} }
\newcommand{\g}{\mathcal{G} }
\newcommand{\p}{\partial}
\newcommand{\half}{\frac{1}{2} }
\newcommand{\T}{\mathcal{T} }
\newcommand{\bi}{\begin{itemize}}
\newcommand{\ei}{\end{itemize}}
\newcommand{\beq}{\begin{equation}}
\newcommand{\eeq}{\end{equation}}
\newcommand{\beas}{\begin{align*}}
\newcommand{\eeas}{\end{align*}}
\newcommand{\cO}{\mathcal{O}}
\newcommand{\cF}{\mathcal{F}}
\newcommand{\cL}{\mathcal{L}}
\newcommand{\BS}{\text{BS}}
$$

<font color = "red">Homework is to be done by each student individually.  To receive full credit, you must email a completed copy of this Jupyter notebook to TAs at [topics_in_qf@163.com](mailto:topics_in_qf@163.com) by the due date and time.  All codes must run correctly and solutions must be written up neatly in Markdown/LaTeX format. If you encounter problems with Jupyter notebook, please contact TA [李新宇](mailto:xinyu911@stu.pku.edu.cn) or [林文鑫](mailto:vincent_lin@stu.pku.edu.cn).

## Name: msj

## Delta and delta-gamma hedging

### 1. (25 points)

A portfolio consists of 
- a long position in 500 shares of a nondividend paying stock with spot price $\$20$,
- a short position in 1000 puts struck at $\$25$ and expiring in 3 months on the stock, assumed lognormally distributed with $30\%$ volatility, 
- $\$10,000$ in money account with annual interest rate $4\%$ continuously compounding.

Answer the following questions.

* (a) What is the value of the portfolio?
* (b) How do you adjust the holdings of stock shares and cash amounts in the portfolio in order to make it delta neutral without changing the postion in puts?
* (c) How do you adjust the portfolio in order to make it delta-gamma neutral by adding position in calls struck at $\$30$? Position in puts cannot be altered. 
* (d) A month later the stock goes up to $\$24$. Determine the value of the delta-neuralized portfolio in (b). 
* (e) How do you rebalance the portfolio in (d) so it remains delta neutral?    


You may consider using the code provided in the cell below for the calculation of deltas and gammas of call and put. 

In [1]:
# as always, import required modules and functions
import numpy as np
from numpy import sqrt, log, exp
import matplotlib.pyplot as plt
import scipy.stats as ss
from scipy.stats import norm
import seaborn as sns

In [2]:
# Black-Scholes formulas
# call
def bs_call(s, K, sigma, t, r=0, d=0):
    d1 = (log(s/K) + (r - d)*t)/(sigma*sqrt(t)) + sigma*sqrt(t)/2
    d2 = d1 - sigma*sqrt(t)
    
    c = s*exp(-d*t)*norm.cdf(d1) - K*exp(-r*t)*norm.cdf(d2)
    delta = exp(-d*t)*norm.cdf(d1)
    gamma = norm.pdf(d1)/s/sigma/sqrt(t)
    
    return {'c': c, 'delta': delta, 'gamma': gamma}

#put
def bs_put(s, K, sigma, t, r=0, d=0):
    d1 = (log(s/K) + (r - d)*t)/(sigma*sqrt(t)) + sigma*sqrt(t)/2
    d2 = d1 - sigma*sqrt(t)
    
    p = K*exp(-r*t)*norm.cdf(-d2) - s*exp(-d*t)*norm.cdf(-d1)
    delta = -exp(-d*t)*norm.cdf(-d1)
    gamma = norm.pdf(d1)/s/sigma/sqrt(t)
    
    return {'p': p, 'delta': delta, 'gamma': gamma}

## <font color=blue> Solution 1. </font>

Parameters: 

$S_0=20$, $K_P=25$, $T=0.25$, $\sigma=0.30$, $r=0.04$.

Initial portfolio:

- $N_S=500$ shares
- $N_P=-1000$ put options ("-" means short)
- $B_0=10000$ in cash


### (1a)
The portfolio value is
$$
V_0=N_S S_0+B_0+N_P P_0.
$$

So the value of the portfolio is $15132.15 


In [3]:
# (1a) 


# parameters
S0 = 20
K_P = 25
T = 0.25
r = 0.04
sigma = 0.30
N_S = 500 # long 500 stock shares
N_P = -1000 # short 1000 puts
cash = 10000

# price and greeks of one put at 0
put = bs_put(S0, K_P, sigma, T, r)
P0 = put['p']
delta_P0 = put['delta']
gamma_P0 = put['gamma']

# total value
value=N_S*S0+cash+N_P*P0

print(f"The value of the portfolio: ${value:.2f}")


The value of the portfolio: $15132.15


### (1b)
Delta of the portfolio: $\Delta_{\Pi} = N_S + N_P \cdot \Delta_P$.

Put delta $\Delta_{P,0} \approx -0.91084$.

For delta neutrality: $N_S^{DN} = -N_P \cdot \Delta_{P,0} = -(-1000) \times (-0.91084) = -910.84\text{ shares}$

Thus, 
- Stock position: -910.84 shares
- Cash balance: $38216.84


In [4]:
# (1b)


# delta neutral
N_S_dn = -N_P * delta_P0
cash_dn = cash + (N_S - N_S_dn) * S0

print(f"Stock position: {N_S_dn:.2f} shares")
print(f"Cash balance: ${cash_dn:.2f}")

Stock position: -910.84 shares
Cash balance: $38216.84


### (1c)

consider calls with strike $K_C=30$

For 1 call, price and Greeks are $C_0\approx0.004752$, $\Delta_{C,0}\approx0.005212$ & $\Gamma_{C,0}\approx0.005001$

The put gamma is $\Gamma_{P,0}\approx0.053753$. 

Gamma neutrality requires $N_P\Gamma_{P,0}+N_C\Gamma_{C,0}=0$, so 
$$ 
N_C=-\frac{N_P\Gamma_{P,0}}{\Gamma_{C,0}} =10747.49. 
$$ 

Then, $ N_S^{DGN}=-966.86. $ 

The adjusted cash balance is $$ B^{DGN}=B_0+(N_S-N_S^{DGN})S_0-N_CC_0 = 39286.10. $$

Thus, 

- Number of calls to buy: 10747.49
- Stock position: -966.86 shares
- Cash balance: $39286.10

In [5]:
# (1c)
# Price and Greeks of one strike-30 call
K_C = 30
call = bs_call(S0, K_C, sigma, T, r)
C0 = call["c"]
delta_C0 = call["delta"]
gamma_C0 = call["gamma"]

# Gamma neutral
N_C = -(N_P * gamma_P0) / gamma_C0
N_S_dgn = - (N_P * delta_P0 + N_C * delta_C0)
# update cash
cash_dgn = cash + (N_S - N_S_dgn) * S0 - N_C * C0

print(f"Number of calls to buy: {N_C:.2f}")
print(f"Stock position: {N_S_dgn:.2f} shares")
print(f"Cash balance: ${cash_dgn:.2f}")

Number of calls to buy: 10747.49
Stock position: -966.86 shares
Cash balance: $39286.10


### (1d)

After 1 month, the new put price is about 1.66. 

The cash account from 1b is now $ B_1=B^{DN}e^{r/12} =38216.84e^{0.04/12} =38344.44$

Following the same formula as in 1a, we get the solution: 
Portfolio Value = $14829.02

In [6]:
# (1d)


# A month later
time=1/12
t_new = T - time

# put
put_new = bs_put(24, K_P, sigma, t_new, r)
P_new = put_new['p']
delta_P_new = put_new['delta']
print(f"New put price: ${P_new:.2f}")

# Cash
cash_new = cash_dn * exp(r * time)

# Portfolio value
V_d = N_S_dn * 24 + cash_new + N_P * P_new

print(f"Portfolio value = ${V_d:.2f}")

New put price: $1.66
Portfolio value = $14829.02


### (1e)

The new put delta is $\Delta_{P,1}\approx-0.58615$

To get delta neutrality, $N_S^{new}+N_P\Delta_{P,1}=0$. Thus, $N_S^{new}=-586.15. $

The required trade is $-586.15-(-910.84)=324.70$, so buy  $\approx324.70$ shares. 
The new cash balance is 
$$ 
B^{new} =B_1-324.70*24 =30551.72. 
$$

After rebalancing, we get New stock position: -586.15 shares & New cash balance: $30551.72

In [7]:
# (1e)


# delta neutral
N_S_new_dn = -N_P * delta_P_new
stock_trade = N_S_new_dn - N_S_dn
cash_rebalance = cash_new - stock_trade *24


direction = "Buy" if stock_trade>= 0 else "Sell"

print(f"New stock position: {N_S_new_dn:.2f} shares")
print(f"{direction} {abs(stock_trade):.2f} shares")
print(f"New cash balance: ${cash_rebalance:.2f}")


New stock position: -586.15 shares
Buy 324.70 shares
New cash balance: $30551.72
